# Solutions 10: Cross-Tokenizer and Representation Distillation

This notebook solves the four exercises of Lab 10. Execution status: all four solutions run
live (the vocabulary-overlap measurement, the truncated sorted-top-p ULD variant with its
proofs, the teacher-layer sweep, and the wrong-teacher control); the training runs inside
exercises 1 and 3 are gated behind `RUN_TRAINING = False`, with the theory prediction or the
expected result stated against the live measurement. Attempt the exercises yourself before
reading this file.

The through-line of all four solutions is the lab's own discipline: prove a measurement on
cases with known answers before letting it near a conclusion, and know a metric's blind spots
before trusting its verdict.

In [1]:
import sys, os, json, math, time
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # everything needed is already cached
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_pipeline import set_seed_everywhere, uld_sorted_loss

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()
print(f"torch {torch.__version__} | RUN_TRAINING: {RUN_TRAINING}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0+cpu | RUN_TRAINING: False


## Exercise 1: Hybrid sweep, starting with the overlap it depends on

**The exercise, restated.** With `uld_use_hybrid_loss=True`, tokens that exist byte-identically
in both vocabularies get JSD (the symmetric, bounded relative of KL that can compare matched
tokens directly) while only the unmatched remainder uses ULD. Measure the exact overlap
fraction between the SmolLM2 and Qwen2.5 vocabularies, then compare hybrid against pure-ULD
runs. The theory to test: hybrid's edge grows with overlap.

**The approach.** The measurement runs live; the two training runs are gated. The overlap is
a set intersection over vocabulary *strings*: both tokenizers are byte-level BPE (they build
tokens from bytes, and they spell each token as a string in the same convention, with a
marker character standing for a leading space), so two tokens match exactly when their
strings are equal. The number to report is the intersection size divided by each vocabulary's
size, because the two vocabularies differ by 3x in size and the fraction therefore depends on
whose denominator you use. The fraction that matters for the hybrid loss is the student-side
one: during training the hybrid split asks, for each position, how much of the *student's*
probability mass sits on tokens the teacher's vocabulary also spells, so the share of the
student vocabulary that is matched is the lever arm of the whole mechanism. The cell also
reloads the lab's measured cross-family ULD baseline so the theory prediction can be stated
against real numbers rather than hope.

In [2]:
tok_s = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
tok_q = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

vs = set(tok_s.get_vocab().keys())
vq = set(tok_q.get_vocab().keys())
inter = vs & vq
frac_s = len(inter) / len(vs)          # share of the student vocab that is matched
frac_q = len(inter) / len(vq)          # share of the teacher vocab that is matched
print(f"SmolLM2 vocab {len(vs):,} | Qwen2.5 vocab {len(vq):,} | "
      f"byte-identical intersection {len(inter):,}")
print(f"overlap fraction: {frac_s:.1%} of the student vocab, {frac_q:.1%} of the teacher's")

baseline = json.load(open("../data/lab10_baseline.json"))
print(f"lab A-2 measured cross-family ULD baseline: {baseline['cross_uld_pretraining']:.3f}")
assert 0.0 < frac_q < frac_s < 1.0, \
    "the small vocab should be mostly contained in the big one, never fully"
assert len(inter) > 0.5 * len(vs), \
    "two byte-level BPE English-heavy vocabularies should share most of the smaller one"

if RUN_TRAINING:
    # Two GOLD runs, identical except for the one flag under test:
    #   run A: use_uld_loss=True, uld_use_hybrid_loss=False   (pure ULD)
    #   run B: use_uld_loss=True, uld_use_hybrid_loss=True    (JSD on matched tokens)
    # Both: teacher Qwen2.5-0.5B-Instruct, student SmolLM2-360M, the lab's B-1
    # config otherwise unchanged (GOLD default lr, dataloader_drop_last=True).
    # Compare held-out ULD, top-1-vs-teacher on matched tokens, and generations.
    pass
else:
    print("RUN_TRAINING=False: the hybrid-vs-pure comparison is gated; "
          "the overlap it depends on is measured above")

SmolLM2 vocab 49,152 | Qwen2.5 vocab 151,665 | byte-identical intersection 39,237
overlap fraction: 79.8% of the student vocab, 25.9% of the teacher's
lab A-2 measured cross-family ULD baseline: 0.321
RUN_TRAINING=False: the hybrid-vs-pure comparison is gated; the overlap it depends on is measured above


**Interpretation.** The measurement came out strongly in hybrid's favor before any
training: just under 80% of the student's vocabulary exists byte-identically in the teacher's
(39,237 of 49,152 tokens), while only about a quarter of the teacher's much larger vocabulary
maps back, exactly the asymmetry the assert expects when a 49k vocabulary meets a 152k one
built with the same byte-level convention. Read against the mechanism: on a typical position,
most of the student's probability mass sits on tokens the hybrid loss can match directly, so
hybrid training gets a token-identity-aware signal (JSD) on the large majority of the mass
and falls back to identity-blind ULD only for the remainder.

The theory prediction for the gated runs, stated concretely: with 80% of the student vocab
matched, hybrid should behave much closer to same-tokenizer distillation than to pure ULD,
so expect it to beat pure ULD clearly on generation quality and teacher agreement while
matching it on the ULD metric itself, and expect the lab's Part C failure signature (ULD
falls while generations worsen) to be far less likely under hybrid, because the JSD term
pins the mass to the right tokens. The gated result would *refute* the exercise's theory
only if hybrid failed to beat pure ULD despite this overlap; at 80% matched, that outcome
would point at an implementation problem (for example, matched-token lookup failing on the
marker-character convention) rather than at the theory.

## Exercise 2: Sorted top-p instead of full-sort

**The exercise, restated.** ULD sorts entire vocabularies at every position. Try matching
only the sorted top-p = 0.99 mass (the smallest set of top probabilities summing to 0.99)
with a single bucket holding the remaining tail, which is Lab 02's top-k trick transplanted.
Same quality at a fraction of the sort cost?

**The approach.** This runs fully live: implement the variant, then prove it the way the lab
proved the original, on cases with known answers, before measuring it on real logits. The
implementation selects each position's largest `cap` probabilities (a partial selection of
`cap` entries, far cheaper than ordering all V), keeps the prefix that reaches 0.99 mass,
zeroes the rest, and appends one explicit tail bucket holding the leftover mass, so the
compared vectors remain full probability distributions. The proof obligations: identity gives
0, cross-vocab-size works, and the approximation stays within tolerance of the full-sort ULD.
The tolerance is not a guess: the truncated comparison can only miss L1 contributions from
tail entries, and each side's tail holds at most 1 - p = 0.01 plus whatever the cap failed to
reach, so the gap to full ULD is bounded near 2 x 0.01 = 0.02, and the assert uses 0.05 to
leave room for the cap residual. Then the real measurement: teacher SmolLM2-360M against
student 135M on real eval rows, full ULD versus the variant, with the size and wall-clock
accounting the exercise asks for.

In [3]:
def uld_sorted_topp_loss(s_logits, t_logits, s_mask, t_mask, p=0.99, cap=512, T=1.0):
    # Returns (loss, mean entries actually kept per position).
    losses, kept = [], []
    for b in range(s_logits.shape[0]):
        sp = F.softmax(s_logits[b][s_mask[b]] / T, dim=-1)
        tp = F.softmax(t_logits[b][t_mask[b]] / T, dim=-1)
        n = min(sp.shape[0], tp.shape[0])
        if n == 0:
            continue
        def truncate(v):
            top, _ = v.topk(min(cap, v.shape[-1]), dim=-1)      # partial selection, size cap
            below_p = (top.cumsum(-1) - top) < p                # keep while mass before < p
            kept_v = top * below_p
            tail = (1.0 - kept_v.sum(-1)).clamp_min(0.0)        # one explicit tail bucket
            return kept_v, tail, below_p.sum(-1).float()
        sk, s_tail, s_n = truncate(sp[:n])
        tk, t_tail, t_n = truncate(tp[:n])
        K = max(sk.shape[-1], tk.shape[-1])
        sk = F.pad(sk, (0, K - sk.shape[-1])); tk = F.pad(tk, (0, K - tk.shape[-1]))
        losses.append(((sk - tk).abs().sum(-1) + (s_tail - t_tail).abs()).mean())
        kept.append(torch.cat([s_n, t_n]).mean())
    return torch.stack(losses).mean(), float(torch.stack(kept).mean())

# Proofs on known answers, mirroring the lab's five-property habit.
g = torch.Generator().manual_seed(0)
B, T_, V1, V2 = 2, 12, 1000, 700
z = 4 * torch.randn(B, T_, V1, generator=g)
z2 = 4 * torch.randn(B, T_, V2, generator=g)
m = torch.ones(B, T_, dtype=torch.bool)
ident, _ = uld_sorted_topp_loss(z, z, m, m)
assert float(ident) < 1e-6, "self-distance must be 0"
full_syn = float(uld_sorted_loss(z2, z, m, m))
appr_syn, kept_syn = uld_sorted_topp_loss(z2, z, m, m)
assert abs(float(appr_syn) - full_syn) < 0.05, \
    f"synthetic: approximation off by {abs(float(appr_syn)-full_syn):.4f}"
print(f"synthetic V={V1} vs V={V2}: full {full_syn:.4f} | top-p {float(appr_syn):.4f} "
      f"| mean entries kept {kept_syn:.0f}")

# Real logits: 360M teacher vs 135M student (same family, so ULD is checkable
# against a case where position counts match and only the method is under test).
teacher = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.float32).eval()
student = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.float32).eval()
ev = torch.load("../data/lab03/eval.pt")
ids, mask = ev["input_ids"][:2, :128], ev["mask"][:2, :128]
with torch.no_grad():
    t_lg = teacher(ids).logits[:, :-1]
    s_lg = student(ids).logits[:, :-1]
mk = mask[:, 1:]

t0 = time.time(); full_real = float(uld_sorted_loss(s_lg, t_lg, mk, mk)); t_full = time.time() - t0
t0 = time.time(); appr_real, kept_real = uld_sorted_topp_loss(s_lg, t_lg, mk, mk); t_appr = time.time() - t0
appr_real = float(appr_real)
V = s_lg.shape[-1]
print(f"real 360M vs 135M: full {full_real:.4f} ({t_full:.2f}s, sorts {V:,}-wide) | "
      f"top-p {appr_real:.4f} ({t_appr:.2f}s, selects {512}-wide)")
print(f"mean entries holding 0.99 mass: {kept_real:.0f} of {V:,} "
      f"({V / max(kept_real,1):.0f}x fewer than a full sort touches)")
assert abs(appr_real - full_real) < 0.05, "real-logit approximation must stay within tolerance"
assert 512 < V / 10, "the selection width must be a small fraction of the vocabulary"
assert kept_real < 512, "0.99 mass must fit inside the cap on a real instruct teacher"


synthetic V=1000 vs V=700: full 0.6588 | top-p 0.6587 | mean entries kept 87


real 360M vs 135M: full 0.2880 (1.24s, sorts 49,152-wide) | top-p 0.2880 (0.21s, selects 512-wide)
mean entries holding 0.99 mass: 73 of 49,152 (677x fewer than a full sort touches)


**Interpretation.** The exercise's question ("same quality at a fraction of the sort
cost?") gets a quantified yes. On the synthetic cross-size case the truncated variant lands
within a few thousandths of the full-sort value, and on real logits the printed gap between
full and top-p ULD is far inside the 0.05 tolerance that the 2 x (1 - p) bound predicts. The
accounting shows why it is cheaper: on this real instruct pair, 0.99 of the probability mass
sits in well under a hundred entries per position on average, so the variant does its
per-position sorting work on a 512-wide selection instead of a 49,152-wide sort, several
hundred times fewer entries ordered. The wall-clock line shows a smaller end-to-end speedup
(a few times, not hundreds), and the gap between those two numbers has a plain reason: both
paths still compute the softmax over the full vocabulary, so only the sorting share of the
work shrank. In a training loop where the softmax is already paid for by the loss itself, the
sorting share is the part ULD adds, which is why the entries-touched number is the one that
prices the method. The tail bucket is what makes this honest rather than lossy: both compared
vectors still sum to 1, so the loss remains an L1 between genuine distributions, exactly the
property Lab 02's tail-bucket estimator had over naive renormalisation.

The practical reading for a training loop: the sort is ULD's per-position bottleneck at
Qwen-scale vocabularies (151,936-wide sorts, every position, every step), and this variant
removes it for a bounded, measured approximation error that is an order of magnitude smaller
than the pretraining baseline ULD of 0.32 it would be optimizing against. The failure mode to
watch is a genuinely flat teacher (high temperature, creative domains), where 0.99 mass can
spread past the cap; the third assert is the tripwire for that, and raising the cap is the
fix.

## Exercise 3: The layer-pair map

**The exercise, restated.** For the representation-matching arm (B-2), sweep which teacher
layer is matched to student layer 15: early, middle, late. TinyBERT's uniform map (student
layer k to teacher layer 2k) is not guaranteed optimal for decoder language models; find the
pairing that helps most and note where it sits relative to the depth midpoint.

**The approach.** The training sweep is gated; what runs live is the lab's own Part A-3
instrument pointed at the question, upgraded in two ways the sweep forces. First, the hidden
states come from real eval rows (six rows, 160 tokens, masked completion positions only)
rather than six short sentences, because the lab's quick check had fewer fit points than the
projector has inputs (577 inputs counting the bias), and a linear fit with more knobs than
data points can reproduce its fitting data perfectly without learning anything, which would
make every candidate layer look equally good. Second, the fit is scored on *held-out*
positions: 60% of the masked positions fit the ridge projector, the remaining 40% score it,
so memorisation earns nothing. The comparison across target layers uses the ratio of held-out
projector MSE to held-out mean-predictor MSE, which is scale-free: different layers' hidden
states have different overall sizes, so raw MSE would reward the smallest-activation layer
rather than the most predictable one. The ratio asks "what fraction of this layer's variance
does the linear map fail to explain", and lower is better. The candidate teacher layers are 5,
16, and 27 of the 360M's 32 (early, middle, late); one forward pass per model yields every
layer's states at once, so the sweep costs two forwards total.

In [4]:
ids6, mask6 = ev["input_ids"][:6, :160], ev["mask"][:6, :160]
with torch.no_grad():
    s_hs = student(ids6, output_hidden_states=True).hidden_states
    t_hs = teacher(ids6, output_hidden_states=True).hidden_states

Hs = s_hs[15][mask6]                       # student layer 15, masked positions only
n = Hs.shape[0]
perm = torch.randperm(n, generator=torch.Generator().manual_seed(0))
fit_i, ho_i = perm[:int(0.6 * n)], perm[int(0.6 * n):]
X = torch.cat([Hs, torch.ones(n, 1)], dim=1)
print(f"{n} masked positions: {len(fit_i)} fit the projector, {len(ho_i)} score it")

CANDIDATES = {"early (layer 5)": 5, "middle (layer 16)": 16, "late (layer 27)": 27}
results = {}
for name, L in CANDIDATES.items():
    Ht = t_hs[L][mask6]
    Xf, Yf = X[fit_i], Ht[fit_i]
    W = torch.linalg.lstsq(Xf.T @ Xf + 1.0 * torch.eye(X.shape[1]), Xf.T @ Yf).solution
    mse_proj = float(((X[ho_i] @ W - Ht[ho_i]) ** 2).mean())
    mse_mean = float(((Yf.mean(0) - Ht[ho_i]) ** 2).mean())
    results[name] = mse_proj / mse_mean
    print(f"student layer 15 -> teacher {name:<18}: held-out unexplained fraction "
          f"{results[name]:.3f}  (projector {mse_proj:.2f} / mean {mse_mean:.2f})")

best = min(results, key=results.get)
assert len(results) == 3 and all(math.isfinite(v) for v in results.values()), "sweep must run"
assert all(v < 1.0 for v in results.values()), \
    "every pairing should beat the mean predictor on held-out positions"
print(f"best pairing: student 15 <-> teacher {best} "
      f"(depth midpoint of the 32-layer teacher is layer 16)")
print("the training version of this sweep is gated; this map is its starting grid")

218 masked positions: 130 fit the projector, 88 score it
student layer 15 -> teacher early (layer 5)   : held-out unexplained fraction 0.882  (projector 16.52 / mean 18.73)


student layer 15 -> teacher middle (layer 16) : held-out unexplained fraction 0.495  (projector 13.23 / mean 26.71)


student layer 15 -> teacher late (layer 27)   : held-out unexplained fraction 0.840  (projector 175.45 / mean 208.94)
best pairing: student 15 <-> teacher middle (layer 16) (depth midpoint of the 32-layer teacher is layer 16)
the training version of this sweep is gated; this map is its starting grid


**Interpretation.** The sweep ran, every pairing beat the held-out mean predictor,
and the ranking is decisive: the middle pairing (teacher layer 16) leaves about 0.50 of the
held-out variance unexplained, while early (layer 5) and late (layer 27) leave about 0.88 and
0.84, so the midpoint pairing explains roughly twice the variance either extreme does.
Student layer 15 sits at the midpoint of the 135M's 30 layers, and its best teacher partner
is the teacher's own midpoint, which is exactly where TinyBERT's uniform proportional map
would have pointed for this layer. So the uniform heuristic survives at the midpoint, and the
sweep's value is the two numbers around it: a map that drifted toward late teacher layers (a
plausible-sounding "match the student's middle to the teacher's refined late features"
intuition) would anchor the MSE term to a target nearly as unrelated as the embedding-adjacent
early layers.

For the gated training version, the expected result follows the lab's B-2 logic: the
best-ratio pairing should show the fastest early agreement gains, because the projector
starts from the strongest linear scaffold, and pairings with worse ratios should show weaker
or slower feature-guidance benefits, shading into the Part C shortcut failure (projector
matches norms, not content) as the ratio approaches 1. Sweeping three ridge fits cost
seconds; sweeping three training runs costs three budgets. That price asymmetry is the
solution's real lesson.

## Exercise 4: The wrong-teacher control

**The exercise, restated.** Run the cross-tokenizer pipeline with a worse teacher and confirm
the evaluation detects the downgrade. If the eval cannot see teacher quality through the ULD
pipeline, it is not measuring what you think it is measuring.

**The approach.** The lab proposes Qwen 0.5B-base as the worse teacher; the cached models
here allow a starker and therefore cleaner version of the same control: Qwen2.5-0.5B-Instruct
(the lab's real teacher) versus gpt2 (the 2019 124M model, a much worse teacher by any
generation-quality standard). Two instruments look at the downgrade, chosen to expose what
each can and cannot see. First, the ULD score of each teacher against the 360M student,
which is what the training pipeline itself watches. The lab's property 2 warned that ULD is
identity-blind: it compares confidence *shapes* with token ownership stripped away, so a
teacher confidently right and a teacher confidently wrong can look identical to it. This
control makes that blindness observable rather than theoretical. Second, a quality probe ULD
cannot fool: each teacher scores a small set of gold completions (short factual English
texts), and the score is bits per byte, the total negative log-probability converted to bits
and divided by the text's byte length. Bits per byte is the fair cross-tokenizer unit because
dividing by bytes cancels the tokenizer's granularity: a model that chops text into more,
smaller tokens gets more, easier predictions, so per-token scores would flatter it, while
per-byte scores cannot. The assert is on the direction: the worse teacher must score worse on
the probe.

In [5]:
GOLD_TEXTS = [
    "The capital of France is Paris, and the capital of Japan is Tokyo.",
    "Water freezes at zero degrees Celsius and boils at one hundred degrees at sea level.",
    "A prime number is a whole number greater than one that is divisible only by one and itself.",
]
probe_text = ("The measurement itself is the deliverable: know the loss value before "
              "training so that afterward you know what changed.")

# The student of the lab's B-1 run is the 360M model (the variable named
# `teacher` above, where it played the same-family teacher role for exercise 2).
tok_360 = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
with torch.no_grad():
    ids = tok_360(probe_text, return_tensors="pt")["input_ids"]
    s_probe = teacher(ids).logits[:, :-1]
s_pm = torch.ones(s_probe.shape[:2], dtype=torch.bool)
del teacher, student                     # make room: one candidate teacher in memory at a time

def teacher_report(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
    with torch.no_grad():
        t_ids = tok(probe_text, return_tensors="pt")["input_ids"]
        t_lg = mdl(t_ids).logits[:, :-1]
        uld = float(uld_sorted_loss(s_probe, t_lg, s_pm,
                                    torch.ones(t_lg.shape[:2], dtype=torch.bool)))
        nats, nbytes = 0.0, 0
        for txt in GOLD_TEXTS:
            gi = tok(txt, return_tensors="pt")["input_ids"]
            lp = F.log_softmax(mdl(gi).logits[:, :-1].float(), dim=-1)
            nats += float(-lp.gather(-1, gi[:, 1:].unsqueeze(-1)).sum())
            nbytes += len(txt.encode("utf-8"))
    del mdl
    return uld, nats / math.log(2) / nbytes

uld_q, bpb_q = teacher_report("Qwen/Qwen2.5-0.5B-Instruct")
uld_g, bpb_g = teacher_report("gpt2")
print(f"{'teacher':<28} {'ULD vs student':>14} {'gold bits/byte':>15}")
print(f"{'Qwen2.5-0.5B-Instruct':<28} {uld_q:>14.3f} {bpb_q:>15.3f}")
print(f"{'gpt2 (much worse)':<28} {uld_g:>14.3f} {bpb_g:>15.3f}")
assert bpb_q < bpb_g, \
    "the quality probe must see the downgrade: the better teacher predicts gold text cheaper"
ratio = abs(uld_g - uld_q) / uld_q
print(f"quality probe detects the downgrade: {bpb_g/bpb_q:.2f}x more bits/byte for gpt2")
print(f"ULD separation between the two teachers: {ratio:.0%} of the good teacher's score "
      f"-> {'ULD also separates them here' if ratio > 0.5 else 'ULD barely separates them'}")

teacher                      ULD vs student  gold bits/byte
Qwen2.5-0.5B-Instruct                 0.321           0.458
gpt2 (much worse)                     0.339           0.977
quality probe detects the downgrade: 2.13x more bits/byte for gpt2
ULD separation between the two teachers: 5% of the good teacher's score -> ULD barely separates them


**Interpretation.** The control passed where it must: the quality probe sees the
downgrade unambiguously, with gpt2 paying a bit over twice the bits per byte on the gold
completions that the Qwen teacher pays (0.98 versus 0.46), a gap no reasonable evaluation
could miss. The instructive column is the ULD one. The printed separation line shows ULD
assigning both teachers scores of the same order against the student, and whatever gap it
shows is unattributable: it reflects differences in confidence *shape* (gpt2 is an older,
flatter, differently calibrated model), not differences in correctness, because property 2
guarantees ULD cannot see which tokens carry the mass. Concretely: if gpt2 were confidently
wrong in exactly the positions where Qwen is confidently right, the ULD column could not
change, while the bits-per-byte column would spread even further.

So the exercise's answer has two halves. Yes, the evaluation as a whole detects the
downgrade, but only because it includes a probe denominated in correctness (teacher-scored
gold text) rather than in distributional shape. A pipeline that watched only its own training
loss, ULD, could swap in a drastically worse teacher and keep reporting plausible numbers,
which is precisely the trap the exercise exists to demonstrate. The operational rule to take
into the gated B-1 runs: every cross-tokenizer experiment carries at least one
teacher-quality probe that is independent of the alignment trick, and bits per byte on a
fixed gold set is the cheapest such probe, one forward pass per teacher, no training
required.